In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from utils.formate_matrix_toMLData import * 
from sklearn.model_selection import train_test_split
from models.model_bayes_1122 import *

from torch.distributions import Exponential

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# データ読み込み
# データの格納リスト
states = []
del_t = []
outputs = []

formater = formate_dataMatrix()

# 読み込む対象の拡張子（例: .csv のみに限定）
VALID_EXTENSIONS = (".csv", ".txt")
IGNORED_PREFIXES = ("._", ".DS_Store", "Thumbs.db")

#data_path
parent_path = "/media/user/TRANSCEND/datas/"
dir_list = ["data1", "data2", "data3","data4","data5"]
# dir_list = ["discrete_data1","discrete_data2","discrete_data3","discrete_data4","discrete_data5"]
# ファイルを処理する関数
def process_file(file_path):
    try:
        print("Processing:", file_path)
        with open(file_path, 'rb') as f:
            all_matrix = np.loadtxt(f, delimiter=",")

        tm = matrix_trimer(all_matrix)
        trm = tm.trim_transitionRateMatrix()
        data = tm.trim_data()
        output_vec = np.array(formater.GetOutputVector_byDiagonal(trm))

        # state: shape (2, num_samples_i)
        state = np.stack([data[:, 0], data[:, 1]], axis=0)
        states.append(state)
        del_t.append(data[:, 2])
        outputs.append(output_vec)

    except Exception as e:
        print(f"❌ Skipping file: {file_path} (Reason: {e})")

# ディレクトリ内のファイルを一括処理
def process_all_files_in_directory(directory, func):
    for filename in os.listdir(directory):
        if filename.startswith(IGNORED_PREFIXES):
            continue
        if not filename.endswith(VALID_EXTENSIONS):
            continue

        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path):
            func(file_path)

# 実行

for dir in dir_list:
    process_all_files_in_directory(parent_path + dir, process_file)
    
# process_all_files_in_directory("/Volumes/TRANSCEND/datas/prototype2", process_file)
# データ分割（Train / Val）
train_states, val_states, train_del_t, val_del_t, train_outputs, val_outputs = train_test_split(
    states, del_t, outputs, test_size=0.1, random_state=42
)

# サイズ確認
print(f"Total data size: {len(states)}")
print(f"Training data size: {len(train_states)}")
print(f"Validation data size: {len(val_states)}")


In [ ]:

# データローダー準備
batch_size = 128
train_dataset = varSets_Datasets(train_states, train_del_t, train_outputs)
val_dataset = varSets_Datasets(val_states, val_del_t, val_outputs)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
model = DeepSets_varSets_forDiagnel(device=device).to(device)
optimizer = optim.Adam(model.parameters(), lr = 0.001)

def npe_loss(
    model: nn.Module,
    state: torch.Tensor,      # (B, 2, L_max)
    delta_t: torch.Tensor,    # (B, L_max)
    lengths: torch.Tensor,    # (B,)
    q_true: torch.Tensor,     # (B,3)
) -> torch.Tensor:
    """
    NPE の損失:
        L(theta) = - E[ log p_theta(q_true | X) ]
    """
    log_p = model.log_prob(state, delta_t, lengths, q_true)  # (B,)
    loss = -log_p.mean()
    return loss

criterion = npe_loss


In [ ]:
#訓練ループの設定
# 早期終了の設定
num_epochs = 1000  # 最大エポック数
patience = 10     # 検証損失が改善しない許容エポック
best_val_loss = float("inf")
patience_counter = 0
# loss_func = KLloss()
best_model_path = 'model_weights/mixed_distribution/mixed_bayes_1122.pth'

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for states_batch, del_t_batch, targets_batch, lengths in train_loader:
        #データのGPUへの転送
        states_batch = states_batch.to(device)
        del_t_batch = del_t_batch.to(device)
        targets_batch = targets_batch.to(device)
        lengths = lengths.to(device)
        
        optimizer.zero_grad()
        outs = model(states_batch,del_t_batch, lengths)
        loss = criterion(outs,state=states_batch, delta_t=del_t_batch,lengths=lengths,q_true=targets_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * states_batch.size(0)
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.6f}")

    # 検証ループ
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for val_states_batch, val_del_t_batch, val_targets_batch, val_lengths in val_loader:
            #データのGPUへの転送
            val_states_batch = val_states_batch.to(device)
            val_del_t_batch = val_del_t_batch.to(device)
            val_targets_batch = val_targets_batch.to(device)
            val_lengths = val_lengths.to(device)
        
            val_outs = model(val_states_batch, val_del_t_batch, val_lengths)
            val_loss = criterion(val_outs,state=val_states_batch,delta_t= val_del_t_batch,lengths=val_lengths,q_true=val_targets_batch)
            val_running_loss += val_loss.item() 
    val_epoch_loss = val_running_loss / len(val_dataset)
    print(f"Validation Loss: {val_epoch_loss:.6f}")

    # 検証損失の改善を確認
    if val_epoch_loss < best_val_loss:
        best_val_loss = val_epoch_loss
        patience_counter = 0
        # 最良モデルの保存
        torch.save(model.state_dict(), best_model_path)
        print("Validation loss improved. Model saved.")
    else:
        patience_counter += 1
        print(f"No improvement in validation loss for {patience_counter} epochs.")

    # 早期終了の判定
    if patience_counter >= patience:
        print("Early stopping triggered.")
        break